In [1]:
from math import log, sqrt, exp
from scipy import stats

In [2]:
# Analytical Black-Scholes-Merton (BSM) Formula
def bsm_call_value(S0, K, T, r, sigma):
    ''' Valuation of European call option in BSM model.
    Analytical formula.
    Parameters
    ==========
    S0 : float
    initial stock/index level
    K : float
    strike price
    T : float
    maturity date (in year fractions)
    r : float
    constant risk-free short rate
    sigma : float
    volatility factor in diffusion term
    Returns
    =======
    value : float
    Price of European call option ati time 0
    '''

    S0 = float(S0)
    d1 = (log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    # print(f'{d1 = }')
    d2 = (log(S0 / K) + (r - 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    # print(f'{d2 = }')
    value = (S0 * stats.norm.cdf(d1, 0.0, 1.0)- K * exp(-r * T) * stats.norm.cdf(d2, 0.0, 1.0))
    # stats.norm.cdf --> cumulative distribution function for normal distribution
    # print('N(d1) =', stats.norm.cdf(d1, 0.0, 1.0))
    # print('N(d2) =', stats.norm.cdf(d2, 0.0, 1.0))

    return value

In [3]:
S0 = 542.5; K = 590; r = 0.05; T = 1/12; sig = 0.2160

#S0 = ; K = 20; r = 0.1; T = 0.25; sig = 0.235
call_value = bsm_call_value(S0, K, T, r, sig)
print("call_value =", call_value)

call_value = 1.6741047684486006


In [4]:
# Vega function (derivative of the BSM function above)
def bsm_vega(S0, K, T, r, sigma):
    ''' Vega of European option in BSM model.        '''

    S0 = float(S0)
    d1 = (log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    vega = S0 * stats.norm.pdf(d1, 0.0, 1.0) * sqrt(T)
    return vega

In [5]:
# Implied volatility, need both the call value function and the vega function

# STUDENTS: Please read more on Newton-Raphson method for solving non-linear equations
def bsm_call_imp_vol(S0, K, T, r, C0, sigma_est, it=100):
    ''' Implied volatility of European call option in BSM model, using Newton-Raphson method
    Parameters
    ==========
    S0 : float
    initial stock/index level
    K : float
    strike price
    T : float
    maturity date (in year fractions)
    r : float
    constant risk-free short rate
    C0: float
    market option price
    sigma_est : float
    estimate of impl. volatility
    it : integer,  number of iterations

    Returns
    =======
    simga_est : float
    numerically estimated implied volatility
    '''
    for i in range(it):
        sigma_est -= ((bsm_call_value(S0, K, T, r, sigma_est) - C0)/bsm_vega(S0, K, T, r, sigma_est))

    return sigma_est

In [6]:
# Example in Sec. 15.11 Implied Volatilities. Students: change into your numerics, note that this notebook only compute implied volatilty from call options.
# how to to determine your maturity "T"? first count how many trading days (excluding saturdays and sundays) from now to maturity, then divide by 252 (trading days per year), that is your "T" in years

#S0 = 21; K = 20; r = 0.1; T = 0.25; C0 = 1.875

#S0 = 539.40; K = 540; T = 18/252; C0 = 10; r = 0.05

S0 = 542.5; K = 540; r = 0.05; T = 1/12; C0 = 16.375

sigma_est = 0.10 # just an initial guess, Students: Please try with other intial guesses, with some "bad" initial guess, the Newton-Raphson method might not converge, try with 0.05 and see what happens
bsm_call_imp_vol(S0, K, T, r, C0, sigma_est, 100)

np.float64(0.22307167144542014)

In [7]:
(1.10 + 2.25) / 2

1.675